# Ingredient co-occurrence graph

This notebook builds a weighted co-occurrence graph between ingredients based on
which ingredients appear together in the same recipe.

The output is an edge list that will be reused in later stages of the project.


In [ ]:
# -- Setup --
# Import the libraries needed. All paths used in this notebook are defined here so they can be changed easily if needed.

from pathlib import Path
import pandas as pd
import ast

# Directory containing the processed outputs from the data cleaning stage of the project
OUT_DIR = Path("../data/processed")

# Cleaned recipes table:
# one row per recipe, with a list of cleaned ingredients
RECIPE_INGREDIENTS_PATH = OUT_DIR / "recipe_ingredients.csv"

# Output file for this notebook:
# weighted ingredient co-occurrence edge list
EDGES_PATH = OUT_DIR / "ingredient_cooccurrence_edges.csv"


# Load cleaned recipes


In [ ]:
# Read the cleaned recipes table produced in the previous stage.
# The ingredient lists were written to CSV as strings, so they need to be converted back into Python lists before use.

# Load the table
recipes_df = pd.read_csv(RECIPE_INGREDIENTS_PATH)

# Basic information to confirm the file loaded as expected
print("Loaded recipe_ingredients.csv")
print("Number of recipes:", len(recipes_df))
print("Columns:", list(recipes_df.columns))

# Restore the ingredients column from string representation
# (e.g. "['egg', 'milk']") to an actual list object
recipes_df["ingredients"] = recipes_df["ingredients"].apply(ast.literal_eval)

# Quick visual check of the data
recipes_df.head()


Loaded recipe_ingredients.csv
Number of recipes: 219890
Columns: ['id', 'ingredients', 'ingredients_raw', 'n_clean_ingredients']


,id,ingredients,ingredients_raw,n_clean_ingredients
0,137739,"[honey, mexican, mixed spice, winter squash]","['winter squash', 'mexican seasoning', 'mixed ...",4
1,31490,"[cheese, egg, milk, pizza crust]","['prepared pizza crust', 'sausage patty', 'egg...",4
2,112140,"[beef, cheddar cheese, chili powder, cumin, ki...","['ground beef', 'yellow onions', 'diced tomato...",9
3,59389,"[herb, new potato, parsley, red pepper, red wi...","['spreadable cheese with garlic and herbs', 'n...",7
4,44061,"[apple cider vinegar, mustard, tomato juice]","['tomato juice', 'apple cider vinegar', 'sugar...",3


# What an edge represents (example/optional)


In [ ]:
# -- Example: co-occurrence pairs from a single recipe -- 
# This cell is only for illustration. It shows how a single recipe is turned into a set of ingredient pairs ("edges")
# in the co-occurrence graph.

from itertools import combinations

# Select one recipe to inspect
row_idx = 0

# Ingredient list for this recipe
ingredients = recipes_df.loc[row_idx, "ingredients"]

print("Recipe id:", recipes_df.loc[row_idx, "id"])
print("Ingredients:", ingredients)

# Remove any duplicates within the recipe and sort the list
# so the pair representation is consistent
unique_ingredients = sorted(set(ingredients))
print("Unique ingredients:", unique_ingredients)

# Generate all unordered ingredient pairs
pairs = list(combinations(unique_ingredients, 2))

print("\nCo-occurrence pairs from this recipe:")
for a, b in pairs:
    print(f"({a}, {b})")

print("\nNumber of pairs:", len(pairs))


Recipe id: 137739
Ingredients: ['honey', 'mexican', 'mixed spice', 'winter squash']
Unique ingredients: ['honey', 'mexican', 'mixed spice', 'winter squash']

Co-occurrence pairs (edges) from this recipe:
(honey, mexican)
(honey, mixed spice)
(honey, winter squash)
(mexican, mixed spice)
(mexican, winter squash)
(mixed spice, winter squash)

Number of pairs: 6


# Co-occurrence counts on a small subset


In [ ]:
# -- Co-occurrence counts on a small subset --
# This step runs the co-occurrence logic on a limited number of recipes. The goal is not performance, but to confirm that:
#   - pairs are counted correctly,
#   - each recipe contributes at most one count per pair,
#   - the most frequent pairs look reasonable.

from itertools import combinations
from collections import Counter

# Number of recipes used for this sanity check
SUBSET_N = 1000

# Mapping: (ingredient_a, ingredient_b) -> number of recipes
edge_counts = Counter()

# Process only a small subset of the data
for ingredients in recipes_df["ingredients"].head(SUBSET_N):
    # Ensure each ingredient is only counted once per recipe
    unique_ingredients = sorted(set(ingredients))

    # Generate all unordered pairs and update counts
    for a, b in combinations(unique_ingredients, 2):
        edge_counts[(a, b)] += 1

# Summary of the subset run
print("Subset size (recipes):", SUBSET_N)
print("Number of unique edges found:", len(edge_counts))

# Inspect the most frequent pairs as a quick sanity check
print("\nTop 20 edges in subset:")
for (a, b), count in edge_counts.most_common(20):
    print(f"({a}, {b}) -> {count}")


Subset size (recipes): 1000
Number of unique edges found: 13882

Top 20 edges in subset:
(egg, flour) -> 145
(flour, milk) -> 89
(flour, powder) -> 75
(flour, soda) -> 75
(egg, milk) -> 72
(flour, vanilla) -> 65
(egg, powder) -> 64
(egg, vanilla) -> 61
(egg, soda) -> 58
(cinnamon, flour) -> 46
(cinnamon, egg) -> 45
(garlic clove, tomato) -> 40
(powder, soda) -> 38
(soda, vanilla) -> 37
(milk, powder) -> 32
(milk, vanilla) -> 32
(powder, vanilla) -> 31
(cream, egg) -> 31
(cinnamon, soda) -> 30
(cream, flour) -> 29


# Full co-occurrence edge list


In [ ]:
#  -- Full co-occurrence edge list --
# Build the final ingredient co-occurrence edge list using the full dataset. This is the version that will be saved to disk
# and reused by later stages of the project.
#
# Key points:
# - ingredient pairs are treated as unordered
# - each recipe contributes at most one count per pair
# - the resulting edge weights represent recipe-level co-occurrence

from itertools import combinations
from collections import Counter

# Stores: (ingredient_a, ingredient_b) -> number of recipes
edge_counts = Counter()

# Process all recipes
for ingredients in recipes_df["ingredients"]:
    # Remove any duplicates within a recipe and keep ordering stable
    unique_ingredients = sorted(set(ingredients))

    # Update counts for all unordered ingredient pairs
    for a, b in combinations(unique_ingredients, 2):
        edge_counts[(a, b)] += 1

print("Finished counting co-occurrence edges.")
print("Number of unique edges:", len(edge_counts))


# -- Convert counts to a table and save -- 
# The edge list is stored in a simple tabular format so it can
# be loaded easily by graph and embedding code later on.

edges_df = pd.DataFrame(
    [
        {"ingredient_a": a, "ingredient_b": b, "weight": count}
        for (a, b), count in edge_counts.items()
    ]
)

# Sort by weight so the strongest co-occurrences appear first
edges_df = edges_df.sort_values(by="weight", ascending=False).reset_index(drop=True)

# Quick visual check
edges_df.head()


# Write the edge list to disk
edges_df.to_csv(EDGES_PATH, index=False)

print(f"Saved co-occurrence edges to: {EDGES_PATH}")


Finished counting co-occurrence edges.
Number of unique edges: 419913

Edge list preview:

Saved co-occurrence edges to: ../data/processed/ingredient_cooccurrence_edges.csv
